# Stg Layer — restaurant_video_reviews (Gemini watches the video, no transcript)

Thay `stg/video_transcripts` từ 2026-11-06 (datle): thay vì chép lời rồi cho `ai_query` đọc, **Gemini xem thẳng video YouTube** (gửi URL, Databricks không gọi youtube.com) và trả về **các trường đánh giá** cho từng quán mà `stg/restaurant_mentions` đã trích từ title / description của video đó. Kết quả ghi vào **chính các dòng của `${silver}.stg_restaurant_mentions`** (grain video × quán), không có bảng riêng.

| Cột | Ý nghĩa |
| --- | --- |
| `reviewer_rating` | Điểm reviewer chấm, quy về **thang 10** (NULL nếu không chấm) |
| `reviewer_summary` | Nhận xét của reviewer về quán (1–3 câu, tiếng Việt) |
| `atmosphere` | Không gian / không khí |
| `parking` | Chỗ để xe |
| `service` | Phục vụ |
| `price_range` | Mức giá reviewer nói (vd "30–50k / tô") |
| `would_return` | Reviewer có muốn quay lại / giới thiệu không (NULL nếu không nói) |
| `video_address` | Địa chỉ nghe / thấy trong video; `stg/restaurant_mentions` dùng khi description không có địa chỉ |
| `review_model`, `review_video_seconds`, `reviewed_at` | Model, độ dài video, thời điểm (đặt cho mọi quán của video, kể cả khi Gemini không nói gì về quán đó, để không gửi lại) |

Trường nào video không nói tới thì NULL (prompt cấm đoán).

**Flow:**
1. Video `restaurant_review` có ít nhất một dòng mention `reviewed_at` IS NULL, dài ≤ `MAX_VIDEO_SECONDS` (20 phút). Ưu tiên title tiếng Việt, video 5–20 phút, nhiều view.
2. Preflight model (`model` widget, mặc định gemini-3.5-flash-lite, dự phòng gemini-3.1-flash-lite).
3. Mỗi video 1 lần gọi: prompt kèm danh sách quán (số thứ tự + tên + địa chỉ đã trích), `responseSchema` JSON → 1 object / quán. **Daily cap** `DAILY_VIDEO_SECONDS` (7 h, free tier ~8 h) tính từ `review_video_seconds` của các video đã review hôm nay (giờ VN); tối đa `max_videos` lần gọi / run; lỗi tạm thời (503, 5xx, timeout) thử lại `RETRY_WAITS`, xen kẽ model; 429 thì dừng, không fail.
4. MERGE vào `stg_restaurant_mentions` theo (`video_id`, `restaurant_name_normalized`).

**Free tier only:** key `gemini/api_key` thuộc GCP project **không gắn billing** (vượt hạn mức chỉ trả 429). Chỉ video public. Free tier: Google có thể dùng dữ liệu gửi lên để cải thiện sản phẩm.


In [ ]:
# --- 2026-11-04 (Free Edition daily DBU cap): LLM / Gemini steps run only in the daily 01h run ---
# Job parameter run_llm: "false" in the 3-hourly refresh runs, "true" when project1_auto_search triggers the job.
dbutils.widgets.text("run_llm", "true")
if dbutils.widgets.get("run_llm").strip().lower() != "true":
    dbutils.notebook.exit("skipped: run_llm = false (LLM steps run once a day, 01h)")


In [ ]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))
dbutils.widgets.text("max_videos", "60")  # Gemini calls per run (1 call = 1 video)
MAX_VIDEOS = int(dbutils.widgets.get("max_videos"))
dbutils.widgets.text("video_id", "")  # optional: review only this video (test)
ONLY_VIDEO_ID = dbutils.widgets.get("video_id").strip()
dbutils.widgets.text("model", "gemini-3.5-flash-lite")
MODEL = dbutils.widgets.get("model")
FALLBACK_MODELS = ["gemini-3.1-flash-lite"]  # used for the retries of a transient error

VIDEOS_TABLE = f"{SILVER}.stg_videos_latest"
MENTIONS_TABLE = f"{SILVER}.stg_restaurant_mentions"
DAILY_VIDEO_SECONDS = 7 * 3600   # free tier allows ~8 h of YouTube video per day
MAX_VIDEO_SECONDS = 20 * 60      # ~91 prompt tokens / s: 20 min ≈ 110k tokens, within free-tier tokens per minute
SLEEP_SECONDS = 10               # free-tier requests per minute
MAX_CONSECUTIVE_ERRORS = 3
RETRY_WAITS = [30, 90, 180]      # seconds before each retry of a transient error (503 high demand, 500, timeout)
RUN_BUDGET_SECONDS = 100 * 60    # stop calling after this (task timeout is 2 h)
PREFERRED_MIN_SECONDS = 5 * 60
SNAPSHOT_TZ = "Asia/Ho_Chi_Minh"
REVIEW_FIELDS = ["reviewer_rating", "reviewer_summary", "atmosphere", "parking", "service", "price_range",
                 "would_return", "video_address"]

PROMPT = """Bạn xem video review quán ăn này (hình và lời nói). Video nhắc tới các quán sau:
{restaurants}

Với MỖI quán trong danh sách, trả về một object với "index" = số thứ tự ở trên và các trường:
- reviewer_rating: điểm người review chấm cho quán, quy về thang 10 (vd 4/5 -> 8). null nếu họ không chấm điểm.
- reviewer_summary: nhận xét của người review về quán (món nào ngon / dở, ấn tượng chung), 1-3 câu tiếng Việt.
- atmosphere: không gian, không khí (rộng / chật, máy lạnh, vỉa hè, sạch sẽ, ồn ào...).
- parking: chỗ để xe (có giữ xe, gửi xe bao nhiêu, đậu ô tô được không...).
- service: thái độ phục vụ, tốc độ ra món.
- price_range: mức giá họ nói, ngắn gọn (vd "35-50k/tô").
- would_return: true nếu họ nói sẽ quay lại / giới thiệu quán, false nếu nói không nên tới, null nếu không nói.
- address: địa chỉ của quán nói hoặc hiện trong video (số nhà, đường, phường / quận), null nếu không có.
Chỉ dùng điều video thật sự nói hoặc cho thấy; trường nào không có thì để null, không đoán.
Chỉ trả về JSON."""
STR = {"type": "STRING", "nullable": True}
RESPONSE_SCHEMA = {"type": "ARRAY", "items": {"type": "OBJECT", "properties": {
    "index": {"type": "INTEGER"},
    "reviewer_rating": {"type": "NUMBER", "nullable": True},
    "reviewer_summary": STR, "atmosphere": STR, "parking": STR, "service": STR, "price_range": STR,
    "would_return": {"type": "BOOLEAN", "nullable": True},
    "address": STR,
}, "required": ["index"]}}

print(f"Target: {MENTIONS_TABLE} · model {MODEL} (fallback {FALLBACK_MODELS}) · max {MAX_VIDEOS} videos per run, "
      f"{DAILY_VIDEO_SECONDS // 3600} h video per day")


In [ ]:
# --- Video seconds reviewed today (Vietnam day); failed calls do not count ---
used_today = spark.sql(f"""
    SELECT coalesce(sum(s), 0) AS s FROM (
        SELECT video_id, max(review_video_seconds) AS s FROM {MENTIONS_TABLE}
        WHERE to_date(from_utc_timestamp(reviewed_at, '{SNAPSHOT_TZ}'))
            = to_date(from_utc_timestamp(current_timestamp(), '{SNAPSHOT_TZ}'))
        GROUP BY video_id)
""").collect()[0]["s"]

# --- Videos with restaurants not reviewed yet ---
todo = spark.sql(f"""
    WITH v AS (  -- same duration parsing (ISO 8601) as gold dim_video
        SELECT s.video_id, s.view_count,
               CASE WHEN s.duration IS NOT NULL THEN
                   3600 * COALESCE(CAST(NULLIF(regexp_extract(s.duration, '(\\\\d+)H', 1), '') AS INT), 0)
                 +   60 * COALESCE(CAST(NULLIF(regexp_extract(s.duration, '(\\\\d+)M', 1), '') AS INT), 0)
                 +        COALESCE(CAST(NULLIF(regexp_extract(s.duration, '(\\\\d+)S', 1), '') AS INT), 0)
               END AS duration_seconds,
               s.title RLIKE '(?i)[ăâđêôơưàáạảãầấậẩẫằắặẳẵèéẹẻẽềếệểễìíịỉĩòóọỏõồốộổỗờớợởỡùúụủũừứựửữỳýỵỷỹ]' AS vietnamese_title
        FROM {VIDEOS_TABLE} s
    ),
    m AS (
        SELECT video_id,
               sort_array(collect_list(struct(restaurant_name_normalized, restaurant_name, address, district))) AS restaurants
        FROM {MENTIONS_TABLE}
        GROUP BY video_id
        HAVING count_if(reviewed_at IS NULL) > 0
    )
    SELECT v.video_id, v.duration_seconds, m.restaurants
    FROM m JOIN v ON v.video_id = m.video_id
    WHERE v.duration_seconds > 0 AND v.duration_seconds <= {MAX_VIDEO_SECONDS}
      AND ('{ONLY_VIDEO_ID}' = '' OR v.video_id = '{ONLY_VIDEO_ID}')
    ORDER BY CASE WHEN v.vietnamese_title THEN 0 ELSE 1 END,
             CASE WHEN v.duration_seconds >= {PREFERRED_MIN_SECONDS} THEN 0 ELSE 1 END,
             v.view_count DESC NULLS LAST, v.video_id
""").collect()
print(f"Videos to review: {len(todo)} · video time already used today: {used_today / 3600:.2f} h")


In [ ]:
# --- Call Gemini with the YouTube URL ---
import json
import time

import requests

API_KEY = dbutils.secrets.get("gemini", "api_key")
BASE = "https://generativelanguage.googleapis.com/v1beta"

# --- Preflight: fail fast when the model name is not offered to this key ---
check = requests.get(f"{BASE}/models/{MODEL}", headers={"x-goog-api-key": API_KEY}, timeout=30)
FALLBACK_MODELS = [m for m in FALLBACK_MODELS if m != MODEL and requests.get(
    f"{BASE}/models/{m}", headers={"x-goog-api-key": API_KEY}, timeout=30).status_code == 200]
MODELS = [MODEL] + FALLBACK_MODELS
print(f"Models used: {MODELS}")
if check.status_code != 200:
    listed = requests.get(f"{BASE}/models", headers={"x-goog-api-key": API_KEY},
                          params={"pageSize": 1000}, timeout=30).json().get("models", [])
    flash = sorted(m["name"].removeprefix("models/") for m in listed
                   if "flash" in m["name"] and "generateContent" in m.get("supportedGenerationMethods", []))
    raise RuntimeError(f"Model {MODEL} not available (HTTP {check.status_code}). "
                       f"Set the model widget to one of: {flash}")


TRANSIENT = {500, 502, 503, 504}


def review_with_retry(row):
    """review() plus retries on transient errors (503 high demand, 5xx, network timeout),
    alternating the main model and the fallback model(s)."""
    for attempt, wait in enumerate([0] + RETRY_WAITS):
        model = MODELS[attempt % len(MODELS)]
        if wait:
            print(f"   retry {attempt}/{len(RETRY_WAITS)} of {row['video_id']} with {model} in {wait}s")
            time.sleep(wait)
        out, status, err = review(row, model)
        if out is not None:
            return out, model, status, err
        if status is not None and status not in TRANSIENT:
            break
    return None, None, status, err


def review(row, model):
    listing = "\n".join(
        f"{i}. {r['restaurant_name']}" + (f" — {r['address']}" if r["address"] else "")
        + (f", {r['district']}" if r["district"] and not r["address"] else "")
        for i, r in enumerate(row["restaurants"], 1))
    body = {
        "contents": [{"parts": [
            {"file_data": {"file_uri": f"https://www.youtube.com/watch?v={row['video_id']}"}},
            {"text": PROMPT.format(restaurants=listing)},
        ]}],
        # Low resolution: ~3x fewer tokens per second of video
        "generationConfig": {"mediaResolution": "MEDIA_RESOLUTION_LOW", "temperature": 0,
                             "responseMimeType": "application/json", "responseSchema": RESPONSE_SCHEMA},
    }
    try:
        resp = requests.post(f"{BASE}/models/{model}:generateContent", headers={"x-goog-api-key": API_KEY},
                             json=body, timeout=600)
    except requests.exceptions.RequestException as e:
        return None, None, f"{type(e).__name__}: {str(e)[:200]}"
    if resp.status_code != 200:
        return None, resp.status_code, resp.text[:300]
    parts = (resp.json().get("candidates") or [{}])[0].get("content", {}).get("parts", [])
    try:
        answer = json.loads("".join(p.get("text", "") for p in parts) or "[]")
    except ValueError as e:
        return None, 200, f"bad JSON: {e}"
    by_index = {a.get("index"): a for a in answer if isinstance(a, dict)}
    out = []
    for i, r in enumerate(row["restaurants"], 1):
        a = by_index.get(i, {})
        rating = a.get("reviewer_rating")
        out.append({"video_id": row["video_id"], "restaurant_name_normalized": r["restaurant_name_normalized"],
                    "reviewer_rating": float(rating) if isinstance(rating, (int, float)) and 0 <= rating <= 10 else None,
                    **{k: (a.get(k) or None) for k in ("reviewer_summary", "atmosphere", "parking", "service", "price_range")},
                    "would_return": a.get("would_return") if isinstance(a.get("would_return"), bool) else None,
                    "video_address": a.get("address") or None})
    return out, 200, None


results, api_calls, sent_seconds, errors, consecutive_errors = [], 0, 0, 0, 0
run_start = time.time()
for row in todo:
    if api_calls >= MAX_VIDEOS:
        break
    if time.time() - run_start > RUN_BUDGET_SECONDS:
        print(f"Run time budget ({RUN_BUDGET_SECONDS // 60} min) reached; the rest waits for the next run")
        break
    if used_today + sent_seconds + row["duration_seconds"] > DAILY_VIDEO_SECONDS:
        print("Daily video cap reached; the rest waits for tomorrow")
        break
    if api_calls:
        time.sleep(SLEEP_SECONDS)
    out, model, status, err = review_with_retry(row.asDict(recursive=True))
    api_calls += 1
    if status == 429:
        print(f"HTTP 429 (free-tier limit) at {row['video_id']}; stopping, the rest waits for the next run")
        break
    if status in (400, 401, 403) and "API key" in (err or ""):
        raise RuntimeError(f"Gemini rejected the API key: {err}")
    if out is None:
        errors += 1
        consecutive_errors += 1
        print(f"❌ {row['video_id']}: HTTP {status} {err}")
        if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
            print(f"{consecutive_errors} errors in a row; stopping, the rest waits for the next run")
            break
        continue
    consecutive_errors = 0
    sent_seconds += row["duration_seconds"]
    for r in out:
        r.update(review_model=model, review_video_seconds=row["duration_seconds"])
    results += out
    filled = sum(any(r[k] is not None for k in REVIEW_FIELDS) for r in out)
    print(f"✅ {row['video_id']} ({row['duration_seconds'] // 60} min): {filled}/{len(out)} restaurant(s) with review fields")

print(f"API calls: {api_calls} · restaurants: {len(results)} · errors: {errors} · video sent: {sent_seconds / 60:.0f} min")


In [ ]:
# --- MERGE into stg_restaurant_mentions (video x restaurant) ---
from pyspark.sql.functions import current_timestamp

if results:
    spark.createDataFrame(results, """video_id STRING, restaurant_name_normalized STRING, reviewer_rating DOUBLE,
        reviewer_summary STRING, atmosphere STRING, parking STRING, service STRING, price_range STRING,
        would_return BOOLEAN, video_address STRING, review_model STRING, review_video_seconds INT""") \
        .withColumn("reviewed_at", current_timestamp()).createOrReplaceTempView("reviews_src")
    cols = REVIEW_FIELDS + ["review_model", "review_video_seconds", "reviewed_at"]
    spark.sql(f"""
        MERGE INTO {MENTIONS_TABLE} AS t
        USING reviews_src AS s
        ON t.video_id = s.video_id AND t.restaurant_name_normalized = s.restaurant_name_normalized
        WHEN MATCHED THEN UPDATE SET {", ".join(f"t.{c} = s.{c}" for c in cols)},
            t.address = coalesce(t.address, s.video_address)
    """)
print(f"✅ Merged {len(results)} restaurant review(s) into {MENTIONS_TABLE}")


In [ ]:
# --- Verify ---
stats = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(reviewed_at) AS reviewed, COUNT(reviewer_rating) AS rating,
           COUNT(reviewer_summary) AS summary, COUNT(atmosphere) AS atmosphere, COUNT(parking) AS parking,
           COUNT(service) AS service, COUNT(price_range) AS price, COUNT(would_return) AS would_return,
           COUNT(video_address) AS video_address
    FROM {MENTIONS_TABLE}
""").collect()[0].asDict()
print(f"📊 {MENTIONS_TABLE}: " + " · ".join(f"{k}: {v}" for k, v in stats.items()))
display(spark.sql(f"""
    SELECT video_id, restaurant_name, reviewer_rating, reviewer_summary, atmosphere, parking, service, price_range,
           would_return, video_address
    FROM {MENTIONS_TABLE} WHERE reviewed_at IS NOT NULL ORDER BY reviewed_at DESC LIMIT 20
"""))
print("\n--- stg_restaurant_video_reviews complete ---")
